In [ ]:
import os
import json

from tqdm import tqdm

import shutil
from pathlib import Path
from lca_filesystem import LCAFilesystem


In [ ]:
original_repos_dir = '/mnt/data/shared-data/lca/repos_updated'

lca_dir = '/mnt/data2/shared-data/lca/plcc_data_dir/'


In [ ]:
permissive_licenses = [
    "MIT License",
    "Apache License 2.0",
    "BSD 3-Clause New or Revised License",
    "BSD 2-Clause Simplified License",
]


In [ ]:
len(os.listdir(original_repos_dir))


In [ ]:
with open('/mnt/data/shared-data/lca/updated_repos_list.json', 'r', encoding="latin-1") as f:
    repos_data = json.load(f)


In [ ]:
repos_list = repos_data['items']

In [ ]:
from dataclasses import dataclass
@dataclass
class RepoInfo:
    language: str
    code_lines_language_ratio: float  # language code lines divided by total code lines
    bytes_language_ratio: float
    license: str
    repo_name: str
    num_commits: int
    main_language: str
    main_language_by_lines: str


def get_repo_info_list(repos_list=repos_list, language: str = 'Kotlin') -> list[RepoInfo]:
    info_list = list()
    for repo in repos_list:
        if language in repo['languages']:
            lang_metrics = [lm for lm in repo['metrics'] if lm['language'] == language]
            if len(lang_metrics) < 1:
                continue
            elif len(lang_metrics) > 1:
                print(lang_metrics)
                raise ValueError
            # if 'kotlin' in repo['name'] and repo['name'].replace('/', '__') not in os.listdir('/mnt/data2/shared-data/lca/repos_updated_kt'):
            #     print(kotlin_metrics[0]['codeLines'] / repo['codeLines'], repo['license'], repo['name'].replace('/', '__'))
            # print(kotlin_metrics)
            bytes_language_ratio = repo['languages'][language] / sum(repo['languages'].values())
            metrics_code_lines = {lm['language']: lm['codeLines'] for lm in repo['metrics']}
            main_language_by_lines = max(metrics_code_lines.items(), key=lambda x: x[1])[0]
            info_list.append(
                RepoInfo(language, 
                         lang_metrics[0]['codeLines'] / repo['codeLines'],
                         bytes_language_ratio,
                         repo['license'], 
                         repo['name'].replace('/', '__'),
                         repo['commits'], 
                         repo['mainLanguage'], 
                         main_language_by_lines)
            )
    return info_list
    

In [ ]:
kotlin_repos = get_repo_info_list(language='Kotlin')
python_repos = get_repo_info_list(language='Python')
java_repos = get_repo_info_list(language='Java')

In [ ]:
def filter_repos(repos: list[RepoInfo], permissive_licenses: list[str] = permissive_licenses, min_lang_ratio: float = 0.1):
    if len(set([repo.language for repo in repos])) !=1 :
        raise ValueError('repos must be of one language')
    lang = repos[0].language
    
    permissive_repos = [repo for repo in repos if repo.license in permissive_licenses]

    _is_ratio = lambda x: x.code_lines_language_ratio >= min_lang_ratio or x.bytes_language_ratio >= min_lang_ratio
    
    good_ratio_repos = [repo for repo in permissive_repos if _is_ratio(repo)]
    
    main_lang_repos = [repo for repo in good_ratio_repos if repo.main_language == lang]
    non_main_lang_repos = [repo for repo in good_ratio_repos if repo.main_language != lang]

    return main_lang_repos, non_main_lang_repos
    
    

In [ ]:
for lang in ['Kotlin', 'Java', 'Python']:
    repos = get_repo_info_list(language=lang)
    a, b = filter_repos(repos)
    print(lang, 'Main Lang:', len(a), 'Other:', len(b))
    

## This is the main function for copying repositories

In [ ]:
lang = 'Python'
repos = get_repo_info_list(language=lang)
a, b = filter_repos(repos)

filtered_repos = a + b

lca_filesystem = LCAFilesystem(lca_dir='/mnt/data2/shared-data/lca/plcc_data_dir/', language=lang.lower())

for repo in tqdm(filtered_repos):
    repos_dir = Path('/mnt/data/shared-data/lca/repos_updated')
    destination_dir = lca_filesystem.permissive_repos_dir
    repo_name = repo.repo_name
    if repo_name not in [rrr.name for rrr in repos_dir.iterdir()]:
        print(repo_name)
    repo_path = repos_dir / repo_name 
    dest_path = destination_dir / repo_name
    try:
        shutil.copytree(repo_path, dest_path)
    except shutil.Error as e:
        print(f">>{repo_name}\nError copying files: {e}")


In [ ]:
len(list(lca_filesystem.permissive_repos_dir.iterdir()))

In [ ]:
len(char_lens), sum(char_lens), sum(char_lens) // 3 / 10**9

In [ ]:
_perc_5 = len(char_lens) * 5 // 100
_perc_10_lines = len(lines_lens) * 10 // 100

print(sorted(char_lens)[_perc_5] // 3, sorted(char_lens)[-_perc_5] // 3)
print(sorted(lines_lens)[_perc_10_lines], sorted(lines_lens)[-_perc_10_lines])

In [ ]:
len(lca_filesystem.completion_files)

In [ ]:
cf['stats_after']

In [ ]:
for repo in tqdm(filtered_repos):
    repos_dir = Path('/mnt/data/shared-data/lca/repos_updated')
    destination_dir = lca_filesystem.permissive_repos_dir
    repo_name = repo.repo_name
    if repo_name not in [rrr.name for rrr in repos_dir.iterdir()]:
        print(repo_name)
    repo_path = repos_dir / repo_name 
    dest_path = destination_dir / repo_name
    shutil.copytree(repo_path, dest_path)
    

In [ ]:
lca_filesystem

In [ ]:
[rrr.name for rrr in repos_dir.iterdir()]

In [ ]:
lca_filesystem.permissive_repos_dir

In [ ]:
filtered_repos[0]

In [ ]:
filtered_repos[0]

In [ ]:
for repo in repos_list:
    wr = weird_repos_dict['Python'][0]
    if repo['name'].replace('/', '__') == wr.repo_name:
        print(json.dumps(repo, indent=4))
        break

In [ ]:
len([ri for ri in kotlin_repos if ri.license in permissive_licenses]), len([ri for ri in python_repos if ri.license in permissive_licenses]), len([ri for ri in java_repos if ri.license in permissive_licenses])

In [ ]:
set(kl[1] for kl in kotlin_lines_), permissive_licenses

In [ ]:
perm_repos = ([
    repo for kl, lic, repo in kotlin_lines_ if (kl>=0.1 and any(lic.lower() == per_lic.lower() for per_lic in permissive_licenses))
])

In [ ]:
src_path = Path('/mnt/data2/shared-data/lca/plcc_data_dir/')
dest_path = Path('/mnt/data/shared-data/lca/plcc_data_dir/')

try:
    shutil.copytree(src_path, dest_path)
except shutil.Error as e:
    print(f">>{repo_name}\nError copying files: {e}")